In [7]:
!pip -q install "mcp<2" groq gradio

import os, sys, json, re, html, time, asyncio, threading
from concurrent.futures import Future
from google.colab import userdata
import gradio as gr
from groq import Groq
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

# ---------- logger ----------
LOGS = []
def L(msg):
    line = f"{time.strftime('%H:%M:%S')} | {msg}"
    LOGS.append(line)
    print(line)

# ============================================================
# 1) MCP SERVER (written to a file, launched by the client)
# ============================================================
SERVER_CODE = '''
from mcp.server.fastmcp import FastMCP
mcp = FastMCP("nutrition")

# sample data per 100 g: (kcal, protein_g). Replace with your assignment-2 data if different.
DATA = {
    "chicken breast": (165, 31), "egg": (155, 13), "banana": (89, 1.1),
    "apple": (52, 0.3), "white rice": (130, 2.7), "paneer": (265, 18),
    "milk": (61, 3.2), "almonds": (579, 21), "peanut butter": (588, 25),
    "salmon": (208, 20), "broccoli": (34, 2.8), "potato": (77, 2),
    "oats": (389, 17), "lentils": (116, 9), "bread": (265, 9),
}

def _find(food):
    f = food.lower().strip()
    if not f:
        return None
    if f in DATA:
        return DATA[f]
    return next((v for k, v in DATA.items() if k in f or f in k), None)

@mcp.tool()
def get_calories(food: str, grams: float = 100) -> str:
    """Return the calories (kcal) for a food item and amount in grams (default 100 g)."""
    v = _find(food)
    if not v:
        return f"No data for '{food}'. Available: {', '.join(DATA)}"
    return f"{v[0] * grams / 100:g} kcal in {grams:g} g of {food}"

@mcp.tool()
def get_protein(food: str, grams: float = 100) -> str:
    """Return the protein (g) for a food item and amount in grams (default 100 g)."""
    v = _find(food)
    if not v:
        return f"No data for '{food}'. Available: {', '.join(DATA)}"
    return f"{v[1] * grams / 100:g} g protein in {grams:g} g of {food}"

if __name__ == "__main__":
    mcp.run()
'''
with open("nutrition_server.py", "w") as f:
    f.write(SERVER_CODE)

# ============================================================
# 2) MCP CLIENT + AGENT
# ============================================================
MODEL = "openai/gpt-oss-120b"
llm = Groq(api_key="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")

class MCPClient:
    """Keeps one MCP stdio connection alive on a background thread (so Gradio can use it)."""
    def __init__(self, server_script="nutrition_server.py"):
        self.script, self.tools, self.error = server_script, [], None
        self.q, self.ready = asyncio.Queue(), threading.Event()
        threading.Thread(target=lambda: asyncio.run(self._main()), daemon=True).start()
        self.ready.wait(30)
        if self.error or not self.tools:
            raise RuntimeError(self.error or "MCP connection timed out (run: !cat server.log)")

    async def _main(self):
        self.loop = asyncio.get_running_loop()
        try:
            L(f"Starting MCP server: {sys.executable} {self.script}")
            params = StdioServerParameters(command=sys.executable, args=[self.script])
            with open("server.log", "w") as slog:
                async with stdio_client(params, errlog=slog) as (r, w):
                    L("stdio transport opened (server process started)")
                    async with ClientSession(r, w) as s:
                        init = await s.initialize()
                        info = getattr(init, "serverInfo", None) or getattr(init, "server_info", None)
                        proto = getattr(init, "protocolVersion", None) or getattr(init, "protocol_version", None)
                        L(f"Session initialized | server={getattr(info, 'name', '?')} | protocol={proto}")
                        self.tools = (await s.list_tools()).tools
                        L(f"Discovered {len(self.tools)} tools: {[t.name for t in self.tools]}")
                        self.ready.set()
                        while True:
                            name, args, fut = await self.q.get()
                            t0 = time.time()
                            L(f"→ call_tool {name} args={args}")
                            try:
                                res = await s.call_tool(name, args)
                                out = "".join(c.text for c in res.content if hasattr(c, "text"))
                                L(f"← result ({time.time() - t0:.2f}s): {out}")
                                fut.set_result(out)
                            except Exception as ex:
                                L(f"✖ tool error: {ex!r}")
                                fut.set_exception(ex)
        except Exception as ex:
            L(f"✖ connection error: {ex!r}")
            self.error = repr(ex)
            self.ready.set()

    def call(self, name, args):
        fut = Future()
        self.loop.call_soon_threadsafe(self.q.put_nowait, (name, args, fut))
        return fut.result(timeout=60)

    def llm_tools(self):
        return [{"type": "function", "function": {
            "name": t.name, "description": t.description or "",
            "parameters": getattr(t, "inputSchema", None) or getattr(t, "input_schema", None)}}
            for t in self.tools]

AGENT_SYS = ("You are a Nutrition AI Agent. For calorie or protein questions about a food, ALWAYS call the "
             "MCP tools get_calories / get_protein (pass the grams the user mentions, default 100). "
             "Answer briefly using the tool results. If a tool says there is no data, say so.")

def run_agent(client, question):
    tools = client.llm_tools()
    msgs = [{"role": "system", "content": AGENT_SYS}, {"role": "user", "content": question}]
    trace = []
    for _ in range(5):
        r = llm.chat.completions.create(model=MODEL, messages=msgs, tools=tools,
                                        tool_choice="auto", temperature=0.2)
        m = r.choices[0].message
        if not m.tool_calls:
            return m.content, trace
        msgs.append({"role": "assistant", "content": m.content or "", "tool_calls": [
            {"id": tc.id, "type": "function",
             "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
            for tc in m.tool_calls]})
        for tc in m.tool_calls:
            args = json.loads(tc.function.arguments or "{}")
            L(f"🤖 LLM requested tool: {tc.function.name} {args}")
            out = client.call(tc.function.name, args)
            trace.append((tc.function.name, args, out))
            msgs.append({"role": "tool", "tool_call_id": tc.id, "content": out})
    return "Too many tool calls.", trace

# ============================================================
# 3) CONNECT + CONSOLE DEMO (take screenshots from this output)
# ============================================================
nutri_mcp = MCPClient()
print("\n✅ Connected to MCP server 'nutrition'\nDiscovered tools:")
for t in nutri_mcp.tools:
    print(f"  🔧 {t.name}: {t.description}")
print()

for qq in ["How many calories are in 100g of chicken breast?",
           "How much protein is in 150g of paneer?"]:
    ans, trace = run_agent(nutri_mcp, qq)
    print(f"\n🧑 {qq}")
    for n, a, o in trace:
        print(f"  🔌 MCP call: {n}({a}) -> {o}")
    print(f"🤖 {ans}\n")

# ============================================================
# 4) UI
# ============================================================
e = lambda x: html.escape(str(x))

def css(color):
    return f"""<style>
  .nc, .nc * {{ color:#111 !important; }}
  .nc {{ background:#fff !important; border:2px solid {color}; border-radius:14px; padding:18px; font-family:sans-serif; }}
  .nc small {{ color:#666 !important; }}
  .nc .badge {{ background:{color} !important; display:inline-block; padding:4px 12px; border-radius:20px; margin:8px 0; }}
  .nc .badge, .nc .badge * {{ color:#fff !important; }}
  .nc .row {{ display:flex; gap:8px; flex-wrap:wrap; }}
  .nc .chip {{ flex:1; min-width:90px; background:#f3f4f6 !important; border-radius:10px; padding:10px; text-align:center; }}
  .nc .chip b {{ font-size:18px; }}
  .nc .step {{ background:#eef2ff !important; border-radius:8px; padding:8px; margin:6px 0; font-family:monospace; }}
  .nc h1, .nc h2 {{ margin:6px 0; }}
</style>"""

# ---- Tab 1: agent using MCP tools ----
def ask(question):
    if not question.strip():
        return "<p>Ask something.</p>"
    try:
        answer, trace = run_agent(nutri_mcp, question)
    except Exception as ex:
        return f"<p style='color:red'>Error: {e(ex)}</p>"
    steps = "".join(f"<div class='step'>🔌 {e(n)}({e(a)}) → {e(o)}</div>" for n, a, o in trace) \
            or "<div class='step'>No tool used.</div>"
    return css("#4f46e5") + f"""<div class='nc'>
  <b>🧑 You:</b> {e(question)}<hr>
  <b>MCP tool calls:</b>{steps}<hr>
  <b>🤖 Agent:</b><p>{e(answer)}</p></div>"""

# ---- Tab 2: full LLM analysis + edible check ----
SYSTEM = """You are a nutrition expert agent. Given a food, quantity and optional health note,
reply ONLY with JSON in this schema:
{"food": str, "edible": "yes"|"no"|"caution", "serving": str, "calories": number,
 "protein_g": number, "carbs_g": number, "fat_g": number, "fiber_g": number, "sugar_g": number,
 "vitamins_minerals": [str], "verdict": str, "reason": str, "tips": [str]}
Rules: if the item is not food or is unsafe/poisonous, set edible "no", numbers 0, and explain.
Use "caution" for foods that are risky in some cases (raw, allergens, health note conflict).
Account for the health note if given. Values are approximate for the stated quantity."""

COLORS = {"yes": ("#16a34a", "✅ Safe to eat"),
          "caution": ("#d97706", "⚠️ Eat with caution"),
          "no": ("#dc2626", "❌ Do not eat")}

def ask_model(user):
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]
    try:
        r = llm.chat.completions.create(model=MODEL, messages=msgs, temperature=0.2,
                                        response_format={"type": "json_object"})
    except Exception:
        r = llm.chat.completions.create(model=MODEL, messages=msgs, temperature=0.2)
    return json.loads(re.search(r"\{.*\}", r.choices[0].message.content, re.S).group(0))

def analyze(food, qty, note):
    if not food.strip():
        return "<p>Enter a food name.</p>"
    user = f"Food: {food}\nQuantity: {qty or '1 standard serving'}\nHealth note: {note or 'none'}"
    try:
        d = ask_model(user)
    except Exception as ex:
        return f"<p style='color:red'>Error: {e(ex)}</p>"
    color, label = COLORS.get(d.get("edible", "caution"), COLORS["caution"])
    chip = lambda k, v, u="g": f"<div class='chip'><b>{e(v)}{u}</b><br><small>{k}</small></div>"
    macros = "".join([chip("Protein", d.get("protein_g", 0)), chip("Carbs", d.get("carbs_g", 0)),
                      chip("Fat", d.get("fat_g", 0)), chip("Fiber", d.get("fiber_g", 0)),
                      chip("Sugar", d.get("sugar_g", 0))])
    vit = ", ".join(map(e, d.get("vitamins_minerals", []))) or "-"
    tips = "".join(f"<li>{e(t)}</li>" for t in d.get("tips", []))
    return css(color) + f"""<div class='nc'>
  <h2>{e(d.get('food'))} <small>({e(d.get('serving'))})</small></h2>
  <div class='badge'>{label}</div>
  <h1>🔥 {e(d.get('calories'))} kcal</h1>
  <div class='row'>{macros}</div>
  <p><b>Vitamins & minerals:</b> {vit}</p>
  <p><b>Verdict:</b> {e(d.get('verdict'))}</p>
  <p><b>Why:</b> {e(d.get('reason'))}</p>
  <b>Tips:</b><ul>{tips}</ul>
  <small>AI estimates only, not medical advice.</small></div>"""

# ---- Tab 3: logs ----
def show_logs():
    try:
        server_side = open("server.log").read()
    except Exception:
        server_side = "(no server.log yet)"
    return "\n".join(LOGS) + "\n\n--- server.log ---\n" + server_side

# ---- Layout ----
tools_html = "".join(f"<li><code>{e(t.name)}</code>: {e(t.description)}</li>" for t in nutri_mcp.tools)
status = css("#16a34a") + f"""<div class='nc'><b>✅ Connected to MCP server</b> (nutrition, stdio)<br>
<b>Discovered tools:</b><ul>{tools_html}</ul></div>"""

with gr.Blocks(theme=gr.themes.Soft(), title="Nutrition AI Agent") as demo:
    gr.Markdown("# 🥗 Nutrition AI Agent\nGroq LLM + MCP client/server.")
    gr.HTML(status)
    with gr.Tabs():
        with gr.Tab("💬 Ask (via MCP tools)"):
            q = gr.Textbox(label="Question", placeholder="How many calories are in 100g of chicken breast?")
            b1 = gr.Button("Ask agent", variant="primary")
            out1 = gr.HTML()
            gr.Examples([["How many calories are in 100g of chicken breast?"],
                         ["How much protein is in 150g of paneer?"],
                         ["Calories and protein in 200g of oats?"]], [q])
            b1.click(ask, q, out1)
            q.submit(ask, q, out1)
        with gr.Tab("🔍 Full analysis + edible check"):
            food = gr.Textbox(label="Food", placeholder="e.g. banana, pani puri, raw egg")
            qty = gr.Textbox(label="Quantity (optional)", placeholder="e.g. 2 pieces, 200 g")
            note = gr.Textbox(label="Health note (optional)", placeholder="e.g. diabetic, nut allergy")
            b2 = gr.Button("Analyze", variant="primary")
            out2 = gr.HTML()
            gr.Examples([["Banana", "1 medium", ""], ["White rice", "1 cup", "diabetic"],
                         ["Raw chicken", "100 g", ""], ["Bleach", "", ""]], [food, qty, note])
            b2.click(analyze, [food, qty, note], out2)
            food.submit(analyze, [food, qty, note], out2)
        with gr.Tab("📜 Connection logs"):
            logbox = gr.Textbox(label="Client + server logs", lines=16, interactive=False)
            gr.Button("Refresh logs").click(show_logs, None, logbox)

demo.launch(share=True, debug=True)

10:31:59 | Starting MCP server: /usr/bin/python3 nutrition_server.py
10:31:59 | stdio transport opened (server process started)
10:32:02 | Session initialized | server=nutrition | protocol=2025-11-25
10:32:02 | Discovered 2 tools: ['get_calories', 'get_protein']

✅ Connected to MCP server 'nutrition'
Discovered tools:
  🔧 get_calories: Return the calories (kcal) for a food item and amount in grams (default 100 g).
  🔧 get_protein: Return the protein (g) for a food item and amount in grams (default 100 g).

10:32:02 | 🤖 LLM requested tool: get_calories {'food': 'chicken breast', 'grams': 100}
10:32:02 | → call_tool get_calories args={'food': 'chicken breast', 'grams': 100}
10:32:02 | ← result (0.01s): 165 kcal in 100 g of chicken breast

🧑 How many calories are in 100g of chicken breast?
  🔌 MCP call: get_calories({'food': 'chicken breast', 'grams': 100}) -> 165 kcal in 100 g of chicken breast
🤖 165 kcal in 100 g of chicken breast.

10:32:02 | 🤖 LLM requested tool: get_protein {'food': 

/tmp/ipykernel_2357/3068445279.py:263: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="Nutrition AI Agent") as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://4f34ef5da45762a2fa.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


10:32:19 | 🤖 LLM requested tool: get_protein {'food': 'paneer', 'grams': 150}
10:32:19 | → call_tool get_protein args={'food': 'paneer', 'grams': 150}
10:32:19 | ← result (0.01s): 27 g protein in 150 g of paneer
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://4f34ef5da45762a2fa.gradio.live
